In [1]:
text = """
1. Reasonableness, not correctness — and BBG still fails

The legal test is not whether BBG was right, but whether its actions were reasonable in process and substance. The evidence shows that BBG’s approach fails even this deferential standard. Key decisions were taken on the basis of incomplete, inconsistent, or weakly supported material, and were not tested against obvious alternative explanations or mitigating evidence.

Why this matters: tribunals defer to employers only where the employer has earned that deference through a genuinely reasonable investigation and decision-making process.

2. Evidence fragility: BBG cannot sustain its factual narrative

BBG’s case relies on assertions that are not consistently supported by contemporaneous evidence. Where technical or operational failures are alleged, BBG has:

failed to produce complete or reliable system evidence,

relied on post-hoc interpretation rather than contemporaneous logs,

and declined or failed to disclose material that would objectively corroborate its claims.

This is not a dispute of opinion; it is a failure of proof. Where an employer’s evidential foundation is fragile, reliance on it is not reasonable.

3. Internal inconsistency and contradiction

BBG’s explanations have shifted over time:

between informal discussions, disciplinary stages, and appeal;

between technical rationale and managerial justification;

and between what was known at the time and what is later asserted.

These inconsistencies undermine the credibility of BBG’s stated belief and demonstrate that the conclusion was not the product of a stable, well-founded assessment.

4. Failure to investigate obvious and relevant lines of inquiry

A reasonable employer would have:

properly investigated alternative technical explanations,

sought clarification where evidence was ambiguous,

and engaged with mitigating context rather than dismissing it.

BBG did not do so. The investigation was selective, outcome-driven, and constrained in scope. This alone places the decision outside the range of reasonable responses.

5. Appeal process lacked genuine corrective function

The appeal did not operate as a meaningful safeguard. Instead, it:

relied substantially on the same contested material,

failed to address core defects in the original investigation,

and did not meaningfully reassess the proportionality of dismissal.

An appeal that merely rehearses the original decision does not cure unfairness; it entrenches it.

6. Disproportionality

Even if BBG had some basis for concern, dismissal was a disproportionate response given:

the absence of sustained or proven misconduct,

the availability of lesser sanctions,

and the failure to account properly for context, intent, and operational realities.

A reasonable employer would have considered alternatives. BBG did not.

7. Cumulative effect: the “reasonableness shield” collapses

Individually, some defects might be survivable. Taken together, they are not.

The combination of:

evidential weakness,

investigative gaps,

internal inconsistency,

and a non-corrective appeal means that BBG’s decision cannot properly be characterised as one a reasonable employer could have taken.
"""

In [2]:
import subprocess
import textwrap

ws_text = text

proc = subprocess.run(
    [
        "python",
        "/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py",
        "--out",
        "/home/hello/Projects/Statements/output/Y_inferred.json",
        "--model",
        "mistral-small3.2:latest",
    ],
    input=ws_text,
    text=True,
    capture_output=True,
)

print(proc.stdout)
print(proc.stderr)


{
  "version": "Y_inferred_v2",
  "x_tests": {
    "X1": {
      "name": "Incomplete or Inconsistent Evidence",
      "scope": "GENERAL",
      "definition": "Evidence used to support a decision is incomplete, inconsistent, or weakly supported.",
      "pattern": "IF a decision is based on incomplete, inconsistent, or weakly supported material THEN support X1",
      "required_elements": [
        "incomplete evidence",
        "inconsistent evidence",
        "weakly supported material"
      ],
      "positive_indicators": [
        "incomplete evidence",
        "inconsistent evidence",
        "weakly supported material"
      ],
      "excludes": [
        "correctness of decision",
        "opinion-based dispute"
      ]
    },
    "X2": {
      "name": "Failure to Investigate Alternative Explanations",
      "scope": "GENERAL",
      "definition": "Failure to investigate obvious and relevant alternative explanations or mitigating evidence.",
      "pattern": "IF an employer fail

In [3]:
from pathlib import Path
import subprocess, shlex
import tqdm

NB_DIR = Path.cwd()
OUT_DIR = NB_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If your Y exists in Statements/output, point to it:
Y_SOURCE = Path("/home/hello/Projects/Statements/output/Y_inferred.json")

# Place a local copy next to the notebook (so paths stay notebook-relative)
Y_JSON = OUT_DIR / "Y_inferred.json"
if not Y_JSON.exists():
    Y_JSON.write_text(Y_SOURCE.read_text(encoding="utf-8"), encoding="utf-8")

PASS1_OUT = OUT_DIR / "pass1_results.jsonl"

BASE_DIR = Path("/home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized")
INPUT_FILES = [
    BASE_DIR / "judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_unknown_faiss_chunk_with_reasoning.jsonl",
]

import shlex
from pathlib import Path

PASS1_SCRIPT = Path("/home/hello/Projects/Statements/code/pass1_scan.py")

cmd = [
    "python", str(PASS1_SCRIPT),
    "--y-json", str(Y_JSON),
    "--out", str(PASS1_OUT),
    "--model", "mistral-small3.2:latest",
    "--ollama-url", "http://localhost:11434/api/generate",
    #"--debug-max", "200",
]

cmd += ["--input", *map(str, INPUT_FILES)]  # <-- key change

cmd_str = " ".join(shlex.quote(c) for c in cmd)
print("Running:\n", cmd_str)
!{cmd_str}



Running:
 python /home/hello/Projects/Statements/code/pass1_scan.py --y-json /home/hello/Projects/Statements/runner/output/Y_inferred.json --out /home/hello/Projects/Statements/runner/output/pass1_results.jsonl --model mistral-small3.2:latest --ollama-url http://localhost:11434/api/generate --input /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl /home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized/judgments_unknown_faiss_chunk_with_reasoning.jsonl
scan judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl: 278lines [03:12,  1.45lines/s, skipped=0, written=278]
scan judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl: 428lines [04:31,  1.58lines/s, skipped=0, written=706]
scan judgments_unknown_faiss_chunk_with_reasoning.jsonl: 479lines [05:23, 

In [4]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "output"
PASS1_PATH = OUT_DIR / "pass1_results.jsonl"
Y_PATH = OUT_DIR / "Y_inferred.json"

# --- load pass1 ---
rows = []
with PASS1_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
df = pd.DataFrame(rows)

# --- load Y (for X-name mapping) ---
Y = json.loads(Y_PATH.read_text(encoding="utf-8"))
x_tests = (Y.get("x_tests") or {})
X_NAME = {k: (v.get("name") or "").strip() for k, v in x_tests.items()}

def matched_names(matched):
    if not isinstance(matched, list):
        return ""
    out = []
    for x in matched:
        x = str(x).strip()
        if not x:
            continue
        name = X_NAME.get(x, "")
        out.append(f"{x}: {name}" if name else x)
    return " | ".join(out)

df["matched_X_names"] = df["matched_X"].apply(matched_names) if "matched_X" in df.columns else ""

# filename guess (same logic)
def filename_guess(row):
    fn = row.get("filename")
    if isinstance(fn, str) and fn.strip():
        return fn.strip()
    item_id = row.get("item_id")
    if not isinstance(item_id, str):
        return None
    parts = item_id.split("::")
    return parts[2] if len(parts) >= 3 else None

df["filename_guess"] = df.apply(filename_guess, axis=1)

# flatten evidence snippets into a single readable field
def flatten_snips(snips, max_each=240, max_total=1200):
    if not isinstance(snips, list):
        return ""
    chunks = []
    total = 0
    for s in snips:
        if not isinstance(s, dict):
            continue
        x = str(s.get("x") or "").strip()
        txt = str(s.get("snippet") or "").strip()
        if not txt:
            continue
        txt = txt.replace("\n", " ")
        if len(txt) > max_each:
            txt = txt[:max_each] + "…"
        chunk = f"{x}: {txt}" if x else txt
        if total + len(chunk) > max_total:
            chunks.append("…")
            break
        chunks.append(chunk)
        total += len(chunk)
    return " | ".join(chunks)

df["evidence_snips_flat"] = df["evidence_snippets"].apply(flatten_snips) if "evidence_snippets" in df.columns else ""

# focus: rows with ANY match
matched_df = df[df["matched_X"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy() if "matched_X" in df.columns else df.iloc[0:0].copy()
matched_df = matched_df.sort_values("confidence", ascending=False) if "confidence" in matched_df.columns else matched_df

# columns to export/view (only keep what exists)
cols = [
    "filename_guess",
    "appeal_type",
    "who_appealed",
    "outcome",
    "successful",
    "summary_clean",
    "reasoning_for_index",
    "confidence",
    "matched_X",
    "matched_X_names",
    "evidence_snips_flat",
    "note",
    "source_file",
    "source_field",
    "item_id",
]
cols = [c for c in cols if c in matched_df.columns]

# export
matched_df.to_csv(OUT_DIR / "matched_x_results.csv", columns=cols, index=False)

# view
matched_df.head(10)[cols]


,filename_guess,appeal_type,who_appealed,outcome,successful,summary_clean,reasoning_for_index,confidence,matched_X,matched_X_names,evidence_snips_flat,note,source_file,source_field,item_id
1176,X_v_Y_UKEAT_0322_12_GE.pdf,Substantive,EMPLOYEE,UNKNOWN,False,The Employment Tribunal found that the Claiman...,The core legal issue was whether the Employmen...,0,[X6],X6: Cumulative Effect of Defects,X6: The core legal issue was whether the Emplo...,The record_text mentions the cumulative effect...,judgments_unknown_faiss_chunk_with_reasoning.j...,reasoning_for_index,judgments_unknown_faiss_chunk_with_reasoning.j...
6,Mr_Neil_Gray_v_University_of_Portsmouth_EA-201...,Substantive,EMPLOYEE,UPHELD,True,"The appellant, Neil Gray, was dismissed from h...",The core legal issue was whether the Universit...,0,[X5],X5: Disproportionality of Response,"X5: loyee, was a proportionate means of achiev...",The text mentions insufficient critical evalua...,judgments_claimant_fav_faiss_chunk_with_reason...,reasoning_for_index,judgments_claimant_fav_faiss_chunk_with_reason...
14,Mr_P_Mefful_v_Merton_and_Lambeth_Citizens_Advi...,Substantive,EMPLOYEE,UPHELD,True,"The Claimant, who was disabled due to a should...",The EAT found the ET erred by failing to rigor...,0,[X2],X2: Failure to Investigate Alternative Explana...,X2: y failing to rigorousl,The text mentions the failure to rigorously as...,judgments_claimant_fav_faiss_chunk_with_reason...,reasoning_for_index,judgments_claimant_fav_faiss_chunk_with_reason...
28,Mr_Raymond_Cairns_v_The_Royal_Mail_Group_Ltd__...,Substantive,EMPLOYEE,UPHELD,True,"The claimant, Mr. Raymond Cairns, was employed...",The core legal issues in dispute centered on w...,0,[X5],X5: Disproportionality of Response,X5: er on his dismissal for ill-he,The text mentions the proportionality of the d...,judgments_claimant_fav_faiss_chunk_with_reason...,reasoning_for_index,judgments_claimant_fav_faiss_chunk_with_reason...
46,Mr_W_Davey_v_Harrods_Ltd__2023__EAT_133.pdf,Strike Out,EMPLOYEE,UPHELD,True,The Employment Tribunal had struck out Mr Dave...,The core legal issue was whether the Employmen...,0,[X5],X5: Disproportionality of Response,X5: nd fairness principles when striking out t...,The record_text mentions the strike-out being ...,judgments_claimant_fav_faiss_chunk_with_reason...,reasoning_for_index,judgments_claimant_fav_faiss_chunk_with_reason...
50,Mr_W_Smith_v_Ideal_Shopping_Direct_Ltd_UKEAT_0...,Substantive,EMPLOYEE,UPHELD,True,"The appellant, Mr. Smith, a gay man, was dismi...",The core legal issues in dispute centered on t...,0,[X1],X1: Incomplete or Inconsistent Evidence,X1: ability of dismissal and its m,The record_text mentions 'inadequately address...,judgments_claimant_fav_faiss_chunk_with_reason...,reasoning_for_index,judgments_claimant_fav_faiss_chunk_with_reason...
1038,Rev_Dr_James_George_Hargreaves_v_1__Evolve_Hou...,Strike Out,EMPLOYEE,REMITTED,True,"The appellant, Rev Dr James George Hargreaves,...",The core legal issue was whether the Employmen...,0,[X5],X5: Disproportionality of Response,"X5: due to his scandalous, vexatious, and unre...",The text mentions the high threshold for strik...,judgments_unknown_faiss_chunk_with_reasoning.j...,reasoning_for_index,judgments_unknown_faiss_chunk_with_reasoning.j...
1042,Ringway_Infrastructure_Services_Ltd_v_Mr_T_J_C...,Strike Out,UNKNOWN,UNKNOWN,False,The Employment Appeal Tribunal allowed the emp...,The core legal issues revolved around the proc...,0,"[X1, X2]",X1: Incomplete or Inconsistent Evidence | X2: ...,X1: o postpone a hearing indefinitely due to t...,The record_text explicitly mentions failure to...,judgments_unknown_faiss_chunk_with_reasoning.j...,reasoning_for_index,judgments_unknown_faiss_chunk_with_reasoning.j...
1058,Science_Museum_Group_v_Ms_Jane_Wess_UKEAT_0260...,Substantive,UNKNOWN,UPHELD,False,"The Claimant, Ms. Jane Wess, was unsuccessful ...",The core legal issues in dispute centered on w...,0,"[X1, X2, X3, X4, X5, X6]",X1: Incomplete or Inconsistent